In [2]:
import numpy as np
from andi_datasets.datasets_theory import datasets_theory as DT
from andi_datasets.utils_trajectories import normalize

In [3]:
from keras.models import load_model

# Load model for predicting anomalous exponents
model_exp = load_model('GRU_exponents_fbm_alpha=0.05-0.1_Tmax=30.h5')



N = 100; T = 30; exponents = np.arange(0.1,1,0.05)

dif = np.zeros((2, N*len(exponents)))

for idx, model in enumerate([2, 0]):
    dataset = DT().create_dataset(T = T, N_models = N, exponents = exponents, models = [model], 
                                  load_trajectories = False)
    np.random.shuffle(dataset)
    X_e = normalize(dataset[:, 2:]).reshape(dataset.shape[0], dataset.shape[1]-2, 1) # shape is (BS, T, 1)
    Y_e = dataset[:, 1] 
    
    expo_p = model_exp.predict(X_e)#[X_e, X_e])
    dif[idx] = (expo_p.flatten()-Y_e)

for mod, error in zip(['FBM','ATTM'], np.abs(dif).mean(1)):
    print(mod, f': MAE = {np.round(error,3)}')


[TensorFlow DLL Diagnostic] Analyzing: c:\Users\Gorka\anaconda3\envs\andi\lib\site-packages\tensorflow\python\_pywrap_tensorflow_internal.pyd
[Error] Failed to load _pywrap_tensorflow_common.dll: INITIALIZATION FAILED (0x45A) - The DLL's DllMain returned false.
    Hint: This often happens if your CPU lacks required instructions (like AVX/AVX2)
    or if the Microsoft Visual C++ Redistributable is outdated/missing.


ImportError: Traceback (most recent call last):
  File "c:\Users\Gorka\anaconda3\envs\andi\lib\site-packages\tensorflow\python\pywrap_tensorflow.py", line 74, in <module>
    from tensorflow.python._pywrap_tensorflow_internal import *
ImportError: DLL load failed while importing _pywrap_tensorflow_internal: A dynamic link library (DLL) initialization routine failed.


Failed to load the native TensorFlow runtime.
See https://www.tensorflow.org/install/errors for some common causes and solutions.
If you need help, create an issue at https://github.com/tensorflow/tensorflow/issues and include the entire stack trace above this error message.

In [1]:
# Testing on generated trajectories


In [7]:
import pandas as pd

df = pd.read_csv('ctrw_0.5.csv')[3:]


traj = df.loc[df['LABEL']=='ID0', 'POSITION_X'].values

In [ ]:
traj[:5], normalize(traj)[:5]

(array([ 0.        , -0.32651117, -0.32187437, -0.24590489, -0.53539245]),
 array([ 0.        , -0.75741224, -0.74665621, -0.5704288 , -1.24195689]))

In [12]:
normalize??

Signature: normalize(trajs)
Source:   
def normalize(trajs):    
    '''
    Normalizes trajectories by substracting average and dividing by
    SQRT of their standard deviation.
    
    Parameters
    ----------
    trajs : np.array
        Array of length N x T or just T containing the ensemble or single trajectory to normalize. 
    '''
    # Checking and saving initial shape
    initial_shape = trajs.shape
    if len(trajs.shape) == 1: # single one d trajectory
        trajs = trajs.reshape(1, trajs.shape[0], 1)
    if len(trajs.shape) == 2: # ensemble of one d trajectories
        trajs = trajs.reshape(trajs.shape[0], trajs.shape[1], 1)
        
    trajs = trajs - trajs.mean(axis=1, keepdims=True)
    displacements = (trajs[:,1:,:] - trajs[:,:-1,:]).copy()    
    variance = np.std(displacements, axis=1)
    variance[variance == 0] = 1    
    new_trajs = np.cumsum((displacements/np.expand_dims(variance, axis = 1)), axis = 1)
    initial_zeros = np.expand_dims(np.zeros((new_traj